# subgraph (하위그래프) 사용

In [1]:
# 하나의 노드가 '그래프' 일수도 있다.  즉, 그래프 안에 그래프를 사용할수 있다.
# 하위 그래프는 독립적으로도 잘 동작하는 하나의 워크플로로 구성하고,
# 상위 그래프에서 이를 재사용하는 방식이 바람직.

# 모듈화를 잘하면 테스트도 편해지고, 재사용성, 확장성 용이합니다.

# import

In [2]:
from dotenv import load_dotenv
print(load_dotenv())

import httpx
from geopy.geocoders import Nominatim # pip install geopy
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode
from langchain.chat_models import init_chat_model
from typing import Literal
import json
from IPython.display import Image, display

True


# 도구

In [3]:
def get_coordinates(city_name: str) -> tuple[float, float]:
    """도시 이름을 받아 위도와 경도를 반환합니다."""
    geolocator = Nominatim(user_agent="weather_app_langgraph")
    location = geolocator.geocode(city_name)
    if location:
        return location.latitude, location.longitude
    raise ValueError(f"좌표를 찾을 수 없습니다: {city_name}")


def get_weather(city_name: str) -> str:
    """도시 이름을 받아 해당 도시의 현재 날씨 정보를 반환합니다."""
    print(f"날씨 조회: {city_name}")
    latitude, longitude = get_coordinates(city_name)
    url = f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current_weather=true"
    # response = httpx.get(url)
    # response.raise_for_status()
    # return json.dumps(response.json())
    return "🌐서울 날씨 맑음"

# 기상전문가 subgraph 생성

In [4]:
def create_weather_agent():
    """날씨 관련 질문을 처리하는 날씨 전문자 서브그래프 생성"""

    model = init_chat_model("gpt-4o", model_provider="openai").bind_tools([get_weather])
    
    tool_node = ToolNode([get_weather])

    def call_model(state: MessagesState):
        return {"messages": [model.invoke(state["messages"])]}

    graph = StateGraph(MessagesState)
    graph.add_node("call_model", call_model)
    graph.add_node("tool_node", tool_node)

    graph.add_edge(START, "call_model")
    graph.add_conditional_edges(
        "call_model",
        lambda s: "tool_node" if s['messages'][-1].tool_calls else END,
        {"tool_node": "tool_node", END: END},
    )
    graph.add_edge("tool_node", "call_model")
    
    return graph.compile()

## 하위그래프 동작 확인

In [5]:
app = create_weather_agent()
result = app.invoke({"messages": [HumanMessage("서울 날씨 어때?")]})
result['messages'][-1].content

날씨 조회: 서울


'서울의 현재 날씨는 맑습니다.'

# 메인그래프 생성

In [6]:
# router() 함수는 노드를 반환하며 사용자의 메시지를 읽어서 어디에 요청을 보낼지 결정하는 역할을 합니다. 
# 예제 코드에서는 단순하게 메시지에 "날씨" 혹은 "기온"이라는 키워드가 있으면 기상 전문가에게, 
# 아니면 일반 에이전트가 처리하도록 했습니다. 

def router(state: MessagesState) -> Literal["weather_expert", "general_agent"]:
    query = state['messages'][-1].content.lower()
    if "날씨" in query or "기온" in query:
        print("라우팅: 기상전문가 에게 위임")
        return "weather_expert"

    print("라우팅: 일반 에이전트가 처리")
    return "general_agent"

In [7]:
# 메인그래프
def create_main_agent(weather_subgraph):  # 매개변수로 서브그래프 받음
    """질문을 라우팅하고 처리하는 메인 에이전트 그래프 생성"""

    main_model = init_chat_model("gpt-4o", model_provider="openai")
    
    workflow = StateGraph(MessagesState)

    workflow.add_node("general_agent", lambda s: {"messages": [main_model.invoke(s['messages'])]})
    # subgraph 를 maingraph 의 노드로 추가!
    workflow.add_node("weather_expert", weather_subgraph)
    workflow.add_conditional_edges(
        START,
        router,
        {
            "weather_expert": "weather_expert",
            "general_agent": "general_agent",
        }
    )

    workflow.add_edge("general_agent", END)
    workflow.add_edge("weather_expert", END)

    return workflow.compile()
    

In [9]:
# 실행
print("=== 서브그래프 예제 (기상전문가) ===")
weather_agent = create_weather_agent()  # 서브그래프
main_agent = create_main_agent(weather_agent)

queries = ["성남의 날씨 어때?", "잠은 몇시가 자는게 좋을까?"]

for query in queries:
    print(f"\n🟠--- 질문: {query} ---")

    result = main_agent.invoke({'messages': [HumanMessage(content=query)]})
    print(f"최종 답변: {result['messages'][-1].content}")
    print("-" * 20)


=== 서브그래프 예제 (기상전문가) ===

🟠--- 질문: 성남의 날씨 어때? ---
라우팅: 기상전문가 에게 위임
날씨 조회: 성남
최종 답변: 성남의 날씨는 맑습니다.
--------------------

🟠--- 질문: 잠은 몇시가 자는게 좋을까? ---
라우팅: 일반 에이전트가 처리
최종 답변: 잠자는 최적의 시간은 개인마다 다를 수 있지만, 일반적으로 건강한 수면을 위해 성인은 밤 10시에서 11시 사이에 잠자리에 드는 것이 좋습니다. 이는 대개 7~9시간의 수면 시간을 확보할 수 있게 해줍니다. 다만, 개인의 생활 패턴, 직업, 수면의 질 등도 고려해야 합니다. 중요한 것은 규칙적인 수면 패턴을 유지하고, 일어나는 시간을 일정하게 유지하여 생체리듬을 잘 맞추는 것입니다.
--------------------
